# Retrueval 효율 개선
- 기존 query에 직장인으로 질문하였으나, 문서에는 소득세 기준이 거주자로 되어 있음
- 이에 따라 vector 저장 시 유사성이 낮아 결과의 정확성이 떨어짐
- 키워드를 사전 활영하여 유사성 향상

In [3]:
%pip install docx2txt langchain-community --quiet


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install langchain-text-splitters --quiet


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install langchain-upstage python-dotenv --quiet


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./data/tax_with_markdown.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

/var/folders/7y/234srpsx7r3_325bg8wjj36m0000gn/T/ipykernel_84254/1394546219.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [9]:
%pip install --upgrade --quiet langchain-pinecone \
     langchain-openai langchain 


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = "tax-markdown-index"
pinecone_api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

/Users/psh/.pyenv/versions/llm-application/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* 인덱스 생성 필요 - 현재 과제에서는 pinecone console에서 진행함

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [21]:
from langchain_core.prompts import ChatPromptTemplate

query = "연봉 5000만원인 직장인의 소득세는 얼마인가요?"

prompt = ChatPromptTemplate.from_messages([
    ('system', """소득세법에 대한 질문에 아래 context를 근거로 답변하세요.
질문에 대한 직접적인 문장이 없더라도, 공제 규정과 세율표를 활용해 단계적으로 계산할 수 있다면 계산하여 결과만을 도출해주세요
context에서 전혀 근거를 찾을 수 없을 때만 모른다고 답하세요.

    [Context]
    {context}
    """),
    ('human', '{question}')
])

In [28]:
from langchain_classic.chains import RetrievalQA

retriever=database.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

ai_message = qa_chain.invoke({"query": query})

In [14]:
retriever.invoke(query)

[Document(id='091bd34a-c027-4909-a076-c45c44fbe5e9', metadata={'source': './data/tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에

In [17]:
ai_message['result']

'연봉 5000만원에 대한 소득세를 계산하기 위해 종합소득과세표준에 대한 세율을 적용하겠습니다. \n\n1. 연봉 5000만원은 1,400만원 초과 5,000만원 이하 구간에 속합니다. 따라서, 해당 구간에 대한 소득세를 계산합니다.\n2. 세율 규정에 따르면, 과세표준이 1,400만원을 초과하고 5,000만원 이하인 경우: 84만원 + (1,400만원을 초과하는 금액의 15퍼센트).\n\n3. 5,000만원 - 1,400만원 = 3,600만원\n\n4. 초과 금액 3,600만원의 15퍼센트는: 3,600만원 × 0.15 = 540만원\n\n5. 따라서, 소득세는 84만원 + 540만원 = 624만원\n\n연봉 5000만원의 직장인의 기준 소득세는 624만원입니다. \n\n이 금액은 기본적인 산출세액이며, 근로소득세액공제, 자녀세액공제 등 추가 공제 사항이 반영되지 않은 금액입니다. 공제를 반영하면 실제 납부하는 세액은 달라질 수 있습니다.'

## keyword 사전 활용

In [ ]:
from langchain_core.output_parsers import StrOutputParser

dictionary = ["사람을 나타내는 표현 -> 거주자"]

keyword_prompt = ChatPromptTemplate.from_template(
    f"""사용자의 질문을 보고, 우리의 사전을 참고하여 사용자의 질문을 변경해주세요.
    만약 변경이 필요 없다고 판단되면 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요

    사전 : {dictionary}

    질문 : {{question}}
    """
)

dictionary_chain = keyword_prompt | llm | StrOutputParser()

In [23]:
new_question = dictionary_chain.invoke({"question": query})

In [24]:
new_question

'연봉 5000만원인 거주자의 소득세는 얼마인가요?'

In [29]:
tax_chain = {"query": dictionary_chain} | qa_chain

In [30]:
ai_response = tax_chain.invoke({"question": query})

In [31]:
ai_response

{'query': '연봉 5000만원인 거주자의 소득세는 얼마인가요?',
 'result': '연봉 5,000만원인 거주자의 소득세를 계산하기 위해서는 해당 과세표준에 따라 계산을 해야 합니다.\n\n1. 1,400만원 이하: 과세표준의 6% = 1,400만원 × 6% = 84만원\n2. 1,400만원 초과 5,000만원 이하: 84만원 + (1,400만원을 초과하는 금액의 15%)\n   - 5,000만원 - 1,400만원 = 3,600만원\n   - 3,600만원 × 15% = 540만원   \n\n3. 최종 세금: 84만원 + 540만원 = 624만원\n\n따라서 연봉 5,000만원인 거주자의 소득세는 624만원입니다.'}